# Diagnosis: Where could reranking help?

Find labeled questions with Recall@30 greater than Recall@5 using the rebuilt Qwen retrieval. Counts and example ranks are observed anew.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.

**Reference input scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions and a separate IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** Experiment 1 (Baseline) --> **this notebook Diagnosis** --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


**Recorded reference run:** 10 November 2026. Saved displays belong to that execution. A new Live run computes its own results; use its current tables rather than treating reference findings as predictions.

## 1. Load the embedding-only records

**What:** load the frozen questions and annotated support IDs, then embed each question and retrieve its 30 candidates.

**Why:** make the opportunity concrete and inspectable using fresh question embeddings and the fixed passage index.

Use the project environment and prepared inputs described in [setup.md](../setup.md). All calculations below use embedding ranks. No reranker ordering, answer text, SQuAD or RAGAS score determines membership.

In [1]:
STUDY = 'diagnosis'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")
view.show('questions')


Setting,Value
Embedding model,qwen/qwen3-embedding-4b
Embedding dimensions,2560
Reranker model,qwen/qwen3-reranker-8b
Generator model,Not used in this stage
Candidate / context budget,30 / 5 passages
Eligible questions in this experiment,15585
Generation limit,Not used in this stage


subgroup
    15585
Name: questions, dtype: int64

query_id,question,subgroup,diagnostic
1102432,. what is a corporation?,,False
query_id,question,subgroup,diagnostic
1102431,why did rachel carson write an obligation to endure,,False
1102421,why did the progressive movement fail to advance racial equality quizlet,,False
1102315,why do police need to understand what the forensic scientists do,,False
1101280,do owls eat in the day,,False
1090282,symptoms of a dying mouse,,False
39449,average number of lightning strikes per day,,False
76162,can you burn your lawn with fertilizer,,False
26117,are volvox autotrophic or heterotrophic,,False


In [2]:
def embed_api(questions):
    # Actual OpenAI-style embedding call. Passage vectors remain prepared inputs.
    return client.embeddings.create(**embedding_payload(
        tutorial.retrieval["embedding_model"], questions, tutorial.retrieval["dimensions"],
        instruction=tutorial.retrieval["query_instruction"]))

retrieval_run = tutorial.retrieve(embed_api)
if retrieval_run is not None:
    display(retrieval_run)


retrieve: saved for CURRENT LIVE RUN 20261111T031923Z-04ffb36349d9


{'questions': 15585,
 'candidate_rows': 467550,
 'query_embedding_matrix': 'query_embeddings.npy',
 'candidates': 'retrieved_top30.parquet',
 'passage_embedding_calls': 0,
 'run_id': '20261111T031923Z-04ffb36349d9'}

## 2. Diagnose the frozen validation subset

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

Compare annotated-support coverage at five versus thirty passages across the **15,585-question frozen subset**. In this **recorded reference run**, **10,242** questions have known support labels. Mean Recall@5 is **0.8915**, mean Recall@30 is **0.9910**, and **1,088** questions meet the opportunity rule.

Recall@k is the fraction of a question's annotated supporting passages found in its first k retrieved passages. These are known-label measurements, not exhaustive relevance judgments. The table below summarizes the subset; it does not estimate population averages from opportunity rows.

**Scope:** this is the frozen subset, not the full MS MARCO validation split.

In [3]:
diagnosis_run = tutorial.diagnose()
if diagnosis_run is not None:
    display(diagnosis_run)
view.show('retrieval_summary')


diagnose: saved for CURRENT LIVE RUN 20261111T031923Z-04ffb36349d9


{'annotated_questions': 10242,
 'opportunities': 1088,
 'run_id': '20261111T031923Z-04ffb36349d9'}

diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,opportunity
0.0001,1.0487,0.8915,0.9910,2.6161,0.8982,0.6331,0.6950,0.1062


## 3. Define the opportunity before testing reranking

**What:** select labeled questions where **Recall@30 > Recall@5**.

**Why:** some annotated evidence is already retrieved at ranks 6–30, but would be left out of a five-passage context. Reordering could help; it has not yet been shown to help.

| Embedding-only situation | What it tells us |
|---|---|
| Recall@30 > Recall@5 | Evidence below the cutoff is recoverable by selection or reordering. |
| Recall@30 = Recall@5 = 1 | All annotated support is already in the top five. |
| Recall@30 = Recall@5 < 1 | Missing annotated support is outside the top thirty; reordering cannot recover those missing IDs. |

Query IDs are stable identifiers, not spreadsheet row numbers. `query_row` is the original dataset row position. The rule depends on support annotations and embedding ranks, not on a generated answer.

In [4]:
if MODE != "plan":
    retrieval_metrics = view.retrieval()
    display(retrieval_metrics.groupby("opportunity").size().rename("questions"))


opportunity
False    9154
True     1088
Name: questions, dtype: int64

## 4. Show the actual opportunity questions

**What:** inspect questions with no annotated support in the top five, and questions missing some complementary support.

**Why:** explain what the selection rule means for real queries rather than stop at an aggregate.

These are embedding-only categories. They are not the later gain/unchanged/loss groups. The displayed examples are ordered deterministically by original dataset row; they are illustrations, not representative random samples. The complete Live-run results are in this run’s `retrieval_metrics.csv` and `retrieved_top30.parquet` artifacts.

In [5]:
view.show('opportunities')


query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,opportunity
1028831,what is vrbo booking fee,,False,1,0.0000,1.0000,9.0000,0.0000,0.0000,0.0000,True
query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,opportunity
1028842,what is vitamin k in dog food,,False,1,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,True
1028844,what is visionpro,,False,1,0.0000,1.0000,10.0000,0.0000,0.0000,0.0000,True
1029080,what is url path,,False,1,0.0000,1.0000,7.0000,0.0000,0.0000,0.0000,True
1029126,what is unusual about the spines of sea urchins,,False,1,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,True
1029139,what is universal value,,False,1,0.0000,1.0000,11.0000,0.0000,0.0000,0.0000,True
1029243,daily protein in,,False,1,0.0000,1.0000,11.0000,0.0000,0.0000,0.0000,True
1029256,what is trump net worth,,False,1,0.0000,1.0000,20.0000,0.0000,0.0000,0.0000,True
1029413,what is treatment for sciatic nerve pain,,False,1,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,True


## 5. Verify coverage directly from passage IDs

**What:** join annotated support IDs to the embedding-ranked candidate IDs, then count support in the first five and thirty.

**Why:** verify the recall columns from evidence membership rather than trust copied scores.

The denominator includes **all** annotated support for each question. A support passage outside the top thirty has a missing candidate rank and counts as a miss. Finding one of two annotated passages gives recall 0.5. This audit recomputes coverage from this Live run’s candidate ranks for the annotated questions in the frozen subset.


In [6]:
# Recall is recomputed directly from gold IDs and this run's candidate ranks.
view.show('retrieval')
view.show('audit')


query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,opportunity
101079,cost for framing a house by the square foot,,False,1,1.0000,1.0000,2.0000,1.0000,0.5000,0.6309,False
query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,opportunity
101123,cost for heartworm treatment dogs,,False,1,1.0000,1.0000,4.0000,1.0000,0.2500,0.4307,False
1028754,what is wall street prime rate,,False,1,1.0000,1.0000,2.0000,1.0000,0.5000,0.6309,False
1028755,what is walgreens starting pay,,False,1,1.0000,1.0000,4.0000,1.0000,0.2500,0.4307,False
1028758,what is wa state's tax on liquor purchases,,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False
1028761,what is vulvar winking,,False,1,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,False
1028796,what is vulnerable animals,,False,1,1.0000,1.0000,2.0000,1.0000,0.5000,0.6309,False
1028799,alamo rental car contact number,,False,1,1.0000,1.0000,2.0000,1.0000,0.5000,0.6309,False
1028812,what is vu,,False,1,0.0000,0.0000,Unavailable,0.0000,0.0000,0.0000,False


Check,Status
"Frozen inputs, question IDs and evidence contexts",Verified


## 6. Inspect one question’s cutoff

Inspect an observed opportunity in the new Qwen retrieval. Its annotated support ranks are recomputed.


In [7]:
view.show('contexts')


passage_id,rank,is_known_support,text,similarity_score
p_f446bafbb6b1eedb4db9,1,True,"Find the IMEI/MEID number. The location of this will vary from phone to phone, but the IMEI/MEID number is usually printed on a sticker that is affixed to the phone underneath the battery. If the phone has an IMEI number but you are on a network that uses MEID numbers, ignore the last digit (IMEI is 15 digits, MEID is 14 digits). GSM networks like AT&T and T-Mobile use IMEI numbers. CDMA networks such as Sprint, Verizon, and US Cellular use MEID numbers.",0.7336
passage_id,rank,is_known_support,text,similarity_score
p_ceb444d4a87718de91ee,2,False,"If this is the case, and you did not record the IMEI number beforehand, there are still a number of ways to retrieve the number: On your devices packaging The easiest way to find your IMEI number is by returning to your original phone packaging. 1 IMEI numbers are printed on the original packaging of your device. 2 Check the base and sides for your 15 digit IMEI number.",0.7200
p_16e01af39f1ab9cc5a66,3,False,"IMEI on iPhone and iPad. Enter the phone settings; Then select General In the filed This device you will find the IMEI number. In models iPhone 3G, 3GS, 4 and 4S, you can check the IMEI number on the SIM card socket. In case of CDMA network there is also the MEID number.",0.7048
p_fbac8ec80aec14e774ec,4,False,"If you have an original iPhone, find the serial number and IMEI engraved on the back. iPad and iPod touch Find the serial number for your iPad Pro, iPad, or iPod touch on the back of the device. On an iPad (cellular model), you'll also find the IMEI. The MEID is the first 14 digits of the IMEI.",0.7027
p_ba06e4d846eb6168282d,5,False,If you are looking for a few methods that you can use to find the IMEI number for your Android smartphone then look no further. This guide will show you some simple and common ways of locating the IMEI number for pretty much any smartphone running on the Android Operating System.,0.6685
p_71838a0d93f9593a28b4,6,True,"An IMEI number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network. We can use this number to block a mobile phone from being used by another person or phone company if it has been lost or stolen. Back to Top.",0.6577
p_e00aa5b17af86f587618,7,False,"The IMEI is only used for identifying the device and has no permanent or semi-permanent relation to the subscriber. Instead, the subscriber is identified by transmission of an International mobile subscriber identity (IMSI) number, which is stored on a SIM card that can in theory be transferred to any handset.",0.6175
p_41ac873926ff56ef6e6a,8,False,The Sleep/Wake button is on the right side of the device. The device has a solid-state Home button with Touch ID. There's a True Tone LED flash on the back and a SIM tray on the right side that holds a fourth form factor (4FF) nano-SIM card. The IMEI is etched on the SIM tray. See the tech specs for iPhone 7.,0.5603
p_8515a786329aa27a5573,9,False,"International Mobile Subscriber Identity (IMSI) Definition - What does International Mobile Subscriber Identity (IMSI) mean? An international mobile subscriber identity (IMSI) is a unique number, usually fifteen digits, associated with Global System for Mobile Communications (GSM) and Universal Mobile Telecommunications System (UMTS) network mobile phone users. The IMSI is a unique number identifying a GSM subscriber. This number has two parts.",0.5283


## 7. Hand a fixed sample to targeted reranking

The diagnosis identifies **1,085** opportunity questions using Recall@30 > Recall@5. Experiment 2 samples **500** opportunities and **200** no-gap controls with the configured seed before reranking. It reuses each selected question's exact thirty candidates. The separate effervescent diagnostic does not enter those group averages.


In [8]:
view.show('retrieval_summary')
tutorial.show()


diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,opportunity
0.0001,1.0487,0.8915,0.9910,2.6161,0.8982,0.6331,0.6950,0.1062


**LIVE run `20261111T031923Z-04ffb36349d9` — only this run’s answers, scores and timings.**

,query_id,embedding_recall_at_5,embedding_recall_at_30
subgroup,,,


,query_id,question,subgroup,embedding_recall_at_5,embedding_recall_at_30,run_id
7466,1053665,what is imei and where do i find it,NaN,0.5,1.0,20261111T031923Z-04ffb36349d9


## Optional appendix: verify the live opportunity selection

Recompute the opportunity IDs from this Live run’s retrieval metrics and verify uniqueness. This is a local check; it makes no additional model calls and does not change the frozen inputs of the later experiments.

In [9]:
# Local cross-check of the opportunity rule; no additional model calls.
if MODE != "plan":
    checked = view.retrieval()
    opportunity_ids = checked.loc[checked.opportunity, "query_id"].tolist()
    assert len(opportunity_ids) == len(set(opportunity_ids))
    display({"opportunity_count": len(opportunity_ids), "rule": "Recall@30 > Recall@5"})


{'opportunity_count': 1088, 'rule': 'Recall@30 > Recall@5'}

### Reproducibility

This Live run retains its query embeddings, thirty candidates per question, question-level recall, support annotations and input provenance under its own run ID. The frozen 15,585-question input and prior run artifacts remain available separately.